# 🧩 Malaria Image Preprocessing – Optimized for Logistic Regression

Goal: Convert malaria cell images into numerical features suitable for Logistic Regression, with optimized memory usage, Standardization, and proper train-test splitting.
---

## **Step 1: Import Libraries & Paths**

- Import essential libraries and define the paths to the image folders.
- Supported image extensions are defined to ignore non-image files.

In [ ]:
import cv2
import os
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle

# Paths
parasitized_path = r"C:\Users\dohah\Documents\Projects\Detecting Malaria Model\MachineLearningProject\img_dataset\Parasitized"
uninfected_path = r"C:\Users\dohah\Documents\Projects\Detecting Malaria Model\MachineLearningProject\img_dataset\Uninfected"

# Supported image extensions
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')

## **Step 2: Load, Resize, Grayscale & Normalize Images**
- Load all images from the folders.
- Resize each image to 64×64 pixels.
- Convert to grayscale to reduce features and memory usage.
- Normalize pixel values to [0,1] for initial scaling (before Standardization).

In [ ]:
def load_images_optimized(folder_path, label, target_size=(64,64)):
    images = []
    labels = []
    count = 0
    for root, dirs, files in os.walk(folder_path):
        for filename in tqdm(files, desc=f"Loading {folder_path}"):
            
            # Skip non-images
            if not (filename.lower().endswith(".png") or filename.lower().endswith(".jpg") or filename.lower().endswith(".jpeg")):
                continue
            img_path = os.path.join(root, filename)
            img = cv2.imread(img_path)
            if img is None:
                continue
            # Resize
            img = cv2.resize(img, target_size)
            # Convert to grayscale
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            # --- New Step: Edge Detection ---          need to understand
            # Example using Canny: Adjust thresholds as needed (100, 200 are common starting points)
            img = cv2.Canny(img, threshold1=100, threshold2=200)      # tell 3alia about it

            # Normalize [0,1]
            img = img.astype('float32') / 255.0

            images.append(img)
            labels.append(label)
            count += 1
    return images, labels, count

# Load images
parasitized_images, parasitized_labels, count_p = load_images_optimized(parasitized_path, 1)
uninfected_images, uninfected_labels, count_u = load_images_optimized(uninfected_path, 0)

# Combine images and labels
images = parasitized_images + uninfected_images
labels = parasitized_labels + uninfected_labels

print(f"Parasitized images loaded: {count_p}")
print(f"Uninfected images loaded: {count_u}")
print(f"Total images loaded: {len(images)}")

Loading C:\Users\dohah\Documents\Projects\Detecting Malaria Model\MachineLearningProject\img_dataset\Parasitized: 100%|██████████| 6/6 [00:00<00:00, 746.10it/s]
Loading C:\Users\dohah\Documents\Projects\Detecting Malaria Model\MachineLearningProject\img_dataset\Uninfected: 100%|██████████| 6/6 [00:00<00:00, 880.11it/s]

Parasitized images loaded: 6
Uninfected images loaded: 6
Total images loaded: 12


## **Step 3: Flatten Images**
- Logistic Regression expects a 1D vector per image, not 2D.
- Flatten each 64×64 grayscale image into a 4096-feature vector.

In [15]:
X = np.array([img.flatten() for img in images], dtype='float32')
y = np.array(labels, dtype='int')

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (12, 4096)
y shape: (12,)


## **Step 4: Train-Test Split with Stratification**
- Split the data into training (80%) and testing (20%) sets.
- stratify=y ensures balanced distribution of classes in both sets.

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print("Train set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)

Train set: (9, 4096) (9,)
Test set: (3, 4096) (3,)


## **Step 5: Standardization**
- Apply StandardScaler to center data around mean=0 and std=1, which improves convergence and accuracy for Logistic Regression.
- Fit on training set only and transform test set with the same scaler.

In [17]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("\n✅ Standardization applied!")
print(f"Train mean: {X_train.mean():.6f}, std: {X_train.std():.6f}")
print(f"Test mean: {X_test.mean():.6f}, std: {X_test.std():.6f}")


✅ Standardization applied!
Train mean: 0.000000, std: 0.585677
Test mean: -0.036996, std: 0.535185


In [18]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


## **Step 6: Optional – Class Distribution & Memory Usage**

In [6]:
# Class distribution
unique, counts = np.unique(y_train, return_counts=True)
print("\nTrain Class Distribution:")
for label, count in zip(unique, counts):
    class_name = "Parasitized" if label == 1 else "Uninfected"
    print(f"   {class_name}: {count} ({count/len(y_train)*100:.1f}%)")

# Memory usage
print(f"\nMemory Usage:")
print(f"X_train: {X_train.nbytes / (1024**2):.2f} MB")
print(f"X_test: {X_test.nbytes / (1024**2):.2f} MB")


Train Class Distribution:
   Uninfected: 11023 (50.0%)
   Parasitized: 11023 (50.0%)

Memory Usage:
X_train: 344.47 MB
X_test: 86.12 MB


## **Step 7: Save Preprocessed Data and Scaler**
- Save preprocessed datasets as .npy files.
- Save the StandardScaler as .pkl for future use on new data.

In [7]:
os.makedirs("preprocessed", exist_ok=True)

np.save("preprocessed/X_train.npy", X_train)
np.save("preprocessed/X_test.npy", X_test)
np.save("preprocessed/y_train.npy", y_train)
np.save("preprocessed/y_test.npy", y_test)

# Save the scaler
with open("preprocessed/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("✅ Preprocessed data saved as .npy files and scaler saved as scaler.pkl")

✅ Preprocessed data saved as .npy files and scaler saved as scaler.pkl


In [19]:
y_pred = clf.predict(X_test)
from sklearn.metrics import accuracy_score
print("Accuracy:", accuracy_score(y_test, y_pred))


Accuracy: 0.6666666666666666
